In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px
import plotly.graph_objects as go
zema = ZemaManager()


# `PARAMETERS TO DEFINE`
- If you want to look at the spot value (earliest contract available for each date), put `0 as contract month`

In [0]:
start_date = datetime(2018, 1, 1)
end_date = datetime.today()

contract_month=12

curve='P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu'

email_list=['florian.girardi-ext@ldc.com']

## Looking for a particular curve?
- Run the code below to get a list of the existing curves containing the key words below (you can add keywords by removing the # in front of the lines, `write the keyword in CAPITAL LETTERS`)

In [0]:
# Curve groups
zema.list_curve_groups()


# Curves
p = zema.list_curves()

p = p[p['Curve Name'].str.contains("FOB")]
p = p[p['Curve Name'].str.contains("CORN")]
p = p[p['Curve Name'].str.contains("AR")]
# p = p[p['Curve Name'].str.contains("US")]
# p = p[p['Curve Name'].str.contains("Odessa")]
display(p)

In [0]:
def plot_contract_curve(start_date, end_date, curve_name, contract_month):
    # 1. Load curve
    df = zema.get_curve(curve=curve_name, period=f"{start_date}::{end_date}")

    if "Last" in df["observation"].unique():
        df = df[df["observation"] == "Last"]
    elif "Settle" in df["observation"].unique():
        df = df[df["observation"] == "Settle"]
        
    df = df[['date', 'value', 'contract_year', 'contract_month']].copy()

    # 3. Extract day/month/year
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year

    # 4. Create virtual_date (dummy year 2000)
    df['virtual_date'] = pd.to_datetime(
        {'year': 2000, 'month': df['month'], 'day': df['day']},
        errors='coerce'
    )
    df = df.dropna(subset=['virtual_date']).sort_values(by='virtual_date')

    if contract_month == 0:
        # Spot contract case: pick earliest contract for each real date
        df['contract_date'] = pd.to_datetime(
            dict(year=df['contract_year'], month=df['contract_month'], day=1)
        )
        df = df.sort_values(['date', 'contract_date']).groupby('date').first().reset_index()
        df['Season'] = df['date'].dt.year.astype(str)
    else:
        # 5. Filter for selected contract month
        df = df[df['contract_month'] == contract_month].copy()

        # 6. Build season label
        df['Season'] = df['contract_month'].map(lambda m: pd.to_datetime(str(m), format="%m").strftime("%b").upper()) \
                       + df['contract_year'].astype(str)

        # 7. Adjust virtual_date logic
        def adjust_virtual_date(row):
            # Months before or equal to the contract month belong to the NEXT year
            if row['virtual_date'].month <= contract_month:
                return row['virtual_date'] + pd.DateOffset(years=1)
            else:
                return row['virtual_date'].replace(year=row['virtual_date'].year)
        df['virtual_date'] = df.apply(adjust_virtual_date, axis=1)

        # 8. Drop invalid (zero) values
        df = df[df['value'] != 0]

        # 9. Seasonal filter condition
        df = df[
            (df['date'].dt.month <= contract_month) & (df['date'].dt.year == df['contract_year'])
            | (df['date'].dt.month > contract_month) & (df['date'].dt.year + 1 == df['contract_year'])
        ].sort_values(by='date')

    # 10. Plot setup
    fig = go.Figure()

    colors = px.colors.qualitative.Plotly
    season_list = sorted(df['Season'].unique())
    season_color_map = {season: colors[i % len(colors)] for i, season in enumerate(season_list)}

    # Add traces
    for season in season_list:
        season_data = df[df['Season'] == season]
        fig.add_trace(go.Scatter(
            x=season_data['virtual_date'],
            y=season_data['value'],
            mode='lines',
            name=f"{season}",
            line=dict(color=season_color_map[season], width=2, dash='solid'),
            yaxis="y1"
        ))

    # Add filter buttons
    buttons = []
    for season in season_list:
        visible = [season in trace.name for trace in fig.data]
        buttons.append(dict(label=season, method="update", args=[{"visible": visible}]))

    buttons.insert(0, dict(label="ALL", method="update", args=[{"visible": [True] * len(fig.data)}]))

    fig.update_layout(
        updatemenus=[dict(
            type="buttons",
            direction="left",
            x=0,
            y=1.15,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True,
        )],
        xaxis=dict(
            tickformat='%b',  # Month format like Jan, Feb...
            dtick="M1",
            hoverformat='%d-%b'),
        hovermode="x unified",
    )

    return fig, df


In [0]:
month_dict = {
    1: "January",
    2: "February",
    3: "March",
    4: "April",
    5: "May",
    6: "June",
    7: "July",
    8: "August",
    9: "September",
    10: "October",
    11: "November",
    12: "December",
    0:'Spot'
}

# Assign month name
month_name = month_dict.get(contract_month, "Invalid month")
month_name

In [0]:
prices=plot_contract_curve(start_date, end_date, curve, contract_month)
prices[0].show()
figure_html=prices[0].to_html(full_html=False, include_plotlyjs='cdn')


In [0]:
report = f"""
<!DOCTYPE html>
<html>
<head>
    <title>{month_name} contract prices for {curve}</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>{curve} {month_name} contract</h1>
    <div class="chart-container">{figure_html}</div>

</body>
</html>
"""
report_bytes = report.encode("utf-8")
      
# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=email_list,
    subject=f'Price curve {month_name} {curve}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body='Attached the report',
    attachment={"report.html": report_bytes}
)
